### 0. Summary of datasets + join logic (current state)

This notebook combines three data sources—**parking meters (supply + rules)**, **payment transactions (demand)**, and **space_counts (capacity)**

The number of unique zones successfully joined is **56** (Section 3.2). For reference, the number of unique zone_id in parking_zones is 63 (Section 1.0), the number of unique zone values in payment_transactions is 60 (Section 1.0), and the number of unique zones in space_counts is 67 (Section 2).

#### 0.1 Main joining challenges
- **Key mismatch**: payment zones are "code - name" strings, while parking uses compact zone_id codes (e.g., DOWNTOWN1, 18-CARSO-L).  
  (See **2.1**.)
- **One-to-many / granularity mismatch**: some payment zones are aggregated (e.g., Uptown, Shadyside, Squirrel Hill, Hill District) but parking splits them into multiple subzones (UPTOWN1/2, SHADYSIDE1/2, SQ.HILL1/2, HILL-DIST/-2).  
  (See **2.2**.)
- **Naming inconsistencies**: ordinals (18th), punctuation (&, /), typos (Disctrict), and abbreviations (MT.WASH, LAWRENCEV) prevent direct string joins.  
  (See **2.3**.)
- **space_counts mixed formats**: includes both payment-style strings and parking-style IDs, plus a few zone labels that do not appear in payments.  
  (See **4.1**.)

#### 0.2 Join strategy implemented
To make joins stable and repeatable, we construct two standardized keys:
- **zone_code + zone_name** extracted from payment zones (e.g., "401 - Downtown 1" → 401, "Downtown 1").  
  (See **2.1**.)
- **zone_group and zone_group_norm**:
  - zone_group: merges split parking subzones into a coarser level when needed (e.g., UPTOWN1/2 → UPTOWN).  
  - zone_group_norm: normalized text key (uppercase, ordinal cleanup, punctuation cleanup, typo fixes) used as the primary join key across datasets.  
  (See **2.2–2.3**.)

To preserve compatibility with rule tables keyed by zone_id, we also save a crosswalk:
- **zone_group_to_zone_id**: maps each merged/normalized group back to original parking zone_id values.  
  (See **3.2**.)

#### 0.3 Final outputs to save
- **parking_zones_final**: parking zones with zone_id, zone_group, zone_group_norm, plus supply summaries (meters/locations).  
  (See **3.1–3.2**.)
- **zone_group_to_zone_id**: crosswalk from zone_group_norm to zone_id for joining to parking_rates, parking_hours, parking_meters.  
  (See **3.2**.)
- **payment_transactions_final**: payment transactions augmented with zone_code, zone_name, zone_group_norm.  
  (See **3.3**.)
- **space_counts_final** + **zone_to_zone_group**: space counts plus a mapping from its zone labels to zone_group_norm so it can join to payments and parking zones.  
  (See **4.2**.)

### 1. Parking Zones & Payment Transactions

Let's first connect parking_zones and payment_trasactions

#### 1.0. Datasets overview

In [2]:
import polars as pl
parking_zones = pl.read_csv("cleaned_data/parking_meter_data/parking_zones.csv")
payment_transactions = pl.read_csv("cleaned_data/payment_transaction_cleaned.csv")
space_counts = pl.read_csv("cleaned_data/space_counts_rates_cleaned.csv")

In [3]:
# for parking meter data we used zone_id as the key to connect different tables (zones,meters,hours,rates)
print(f"Number of unique zone_id in parking_zones: {parking_zones.select("zone_id").height}")
print(parking_zones.select("zone_id").sort("zone_id").to_series().to_list())

Number of unique zone_id in parking_zones: 62
['18-CARSO-L', '18-SIDNE-L', '19-CARSO-L', '20-SIDNE-L', '42-BUTLE-L', '5224BUTL-L', 'ALLENTOWN', 'ANSL-BEA-L', 'ASTE-WAR-L', 'BAKERY-SQ', 'BEAC-BAR-L', 'BEECHVIE-L', 'BEECHVIEW', 'BLOOMFIELD', 'BROOKLIN-L', 'BROOKLINE', 'BROW-SAN-L', 'CARRICK', 'DOUG-PHI-L', 'DOWNTOWN1', 'DOWNTOWN2', 'EASTCARS-L', 'EASTLIB', 'EASTOHIO-L', 'EVA-BEAT-L', 'FORB-MUR-L', 'FORB-SHA-L', 'FRIE-CED-L', 'HILL-DIST', 'HILL-DIST-2', 'HOME-ZEN-L', 'IVY-BELL-L', 'JCC-L', 'KNOXVILLE', 'LAWRENCEV', 'MAIN-ALE-L', 'MELONPARK', 'MT.WASH', 'NORTHSHORE', 'NORTHSIDE', 'OAKLAND1', 'OAKLAND2', 'OAKLAND3', 'OAKLAND4', 'OBSERHIL-L', 'PENNC.NW-L', 'SHADYSIDE1', 'SHADYSIDE2', 'SHER-HAR-L', 'SHER-KIR-L', 'SOUTHSIDE', 'SQ.HILL1', 'SQ.HILL2', 'STRIPDIST', 'TAME-BEA-L', 'TAYLOR-L', 'TECHNOLOGY', 'UPTOWN1', 'UPTOWN2', 'W CIRC DR', 'WALT-WAR-L', 'WEST END']


In [4]:
# check zone of payment_transactions
pt_unique_zone = payment_transactions.select("zone").unique().sort("zone")
print(f"Number of unique zone in payment_transactions: {pt_unique_zone.height}")
print(pt_unique_zone.to_series().to_list())

Number of unique zone in payment_transactions: 60
['301 - Sheridan Harvard Lot', '302 - Sheridan Kirkwood Lot', '304 - Tamello Beatty Lot', '307 - Eva Beatty Lot', '308 - Harvard Beatty Lot', '311 - Ansley Beatty Lot', '314 - Penn Circle NW Lot', '321 - Beacon Bartlett Lot', '322 - Forbes Shady Lot', '323 - Douglas Phillips Lot', '324 - Forbes Murray Lot', '325 - JCC/Forbes Lot', '328 - Ivy Bellefonte Lot', '329 - Centre Craig', '331 - Homewood Zenith Lot', '334 - Taylor Street Lot', '335 - Friendship Cedarville Lot', '337 - 52nd & Butler Lot', '338 - 42nd & Butler Lot', '341 - 18th & Sidney Lot', '342 - East Carson Lot', '343 - 19th & Carson Lot', '344 - 18th & Carson Lot', '345 - 20th & Sidney Lot', '351 - Brownsville & Sandkey Lot', '354 - Walter/Warrington Lot', '355 - Asteroid Warrington Lot', '357 - Shiloh Street Lot', '361 - Brookline Lot', '363 - Beechview Lot', '369 - Main/Alexander Lot', '371 - East Ohio Street Lot', '375 - Oberservatory Hill Lot', '401 - Downtown 1', '402 - 

**There are many issues that we should address in order to join these tables:**

#### 1.1. Fix the join key mismatch (payments vs parking)
- Payments identify zones as a single string like "401 - Downtown 1" (numeric code + zone name).  
- Parking uses compact zone identifiers like DOWNTOWN1, SHER-HAR-L, SQ.HILL1.

**Fix:** Split the payment zone field into two clean variables:
- zone_code: the leading numeric code (e.g., 401)
- zone_name: the zone name text after " - " (e.g., "Downtown 1")

This gives us a stable key (zone_code) and a readable label (zone_name) to build a mapping (crosswalk) to parking zone_id.

- "401 - Downtown 1" → zone_code = 401, zone_name = "Downtown 1"

In [5]:
# Separating the zone name from zone code

# payment_transactions
pt = (
    payment_transactions
    .with_columns([
        # Extract numeric code at the start: "401 - Downtown 1" -> 401
        pl.col("zone")
          .cast(pl.Utf8)
          .str.extract(r'^"?(\d+)')  # if you don't have quotes, r'^(\d+)' is enough
          .cast(pl.Int64)
          .alias("zone_code"),

        # Extract zone name after the dash: "401 - Downtown 1" -> "Downtown 1"
        pl.col("zone")
          .cast(pl.Utf8)
          .str.replace(r'^"?(\d+)\s*-\s*', "")  # remove leading '401 - '
          .str.replace('"', "")
          .str.strip_chars()
          .alias("zone_name"),
    ])
)

pt.select(["zone", "zone_code", "zone_name"]).head()

zone,zone_code,zone_name
str,i64,str
"""421 - NorthSide""",421,"""NorthSide"""
"""403 - Uptown""",403,"""Uptown"""
"""412 - East Liberty""",412,"""East Liberty"""
"""421 - NorthSide""",421,"""NorthSide"""
"""402 - Downtown 2""",402,"""Downtown 2"""


In [6]:
# Display unique zone_name of payment transactions
pt_zone_names = pt.select("zone_name").unique()
print(f"Number of unique zone names after extraction: {pt_zone_names.height}")
print(pt_zone_names.sort("zone_name").to_series().to_list())

Number of unique zone names after extraction: 60
['18th & Carson Lot', '18th & Sidney Lot', '19th & Carson Lot', '20th & Sidney Lot', '42nd & Butler Lot', '52nd & Butler Lot', 'Allentown', 'Ansley Beatty Lot', 'Asteroid Warrington Lot', 'Bakery Sq', 'Beacon Bartlett Lot', 'Beechview', 'Beechview Lot', 'Bloomfield (On-street)', 'Brookline', 'Brookline Lot', 'Brownsville & Sandkey Lot', 'Carrick', 'Centre Craig', 'Douglas Phillips Lot', 'Downtown 1', 'Downtown 2', 'East Carson Lot', 'East Liberty', 'East Ohio Street Lot', 'Eva Beatty Lot', 'Forbes Murray Lot', 'Forbes Shady Lot', 'Friendship Cedarville Lot', 'Harvard Beatty Lot', 'Hill District', 'Homewood Zenith Lot', 'Ivy Bellefonte Lot', 'JCC/Forbes Lot', 'Knoxville', 'Lawrenceville', 'Main/Alexander Lot', 'Mellon Park', 'Mt. Washington', 'NorthSide', 'Northshore', 'Oakland 1', 'Oakland 2', 'Oakland 3', 'Oakland 4', 'Oberservatory Hill Lot', 'Penn Circle NW Lot', 'SS & SSW', 'Shadyside', 'Sheridan Harvard Lot', 'Sheridan Kirkwood Lot'

In [7]:
# Remember the unique zone_id from parking_zones
print(f"Number of unique zone_id in parking_zones: {parking_zones.select("zone_id").height}")
print(parking_zones.select("zone_id").sort("zone_id").to_series().to_list())

Number of unique zone_id in parking_zones: 62
['18-CARSO-L', '18-SIDNE-L', '19-CARSO-L', '20-SIDNE-L', '42-BUTLE-L', '5224BUTL-L', 'ALLENTOWN', 'ANSL-BEA-L', 'ASTE-WAR-L', 'BAKERY-SQ', 'BEAC-BAR-L', 'BEECHVIE-L', 'BEECHVIEW', 'BLOOMFIELD', 'BROOKLIN-L', 'BROOKLINE', 'BROW-SAN-L', 'CARRICK', 'DOUG-PHI-L', 'DOWNTOWN1', 'DOWNTOWN2', 'EASTCARS-L', 'EASTLIB', 'EASTOHIO-L', 'EVA-BEAT-L', 'FORB-MUR-L', 'FORB-SHA-L', 'FRIE-CED-L', 'HILL-DIST', 'HILL-DIST-2', 'HOME-ZEN-L', 'IVY-BELL-L', 'JCC-L', 'KNOXVILLE', 'LAWRENCEV', 'MAIN-ALE-L', 'MELONPARK', 'MT.WASH', 'NORTHSHORE', 'NORTHSIDE', 'OAKLAND1', 'OAKLAND2', 'OAKLAND3', 'OAKLAND4', 'OBSERHIL-L', 'PENNC.NW-L', 'SHADYSIDE1', 'SHADYSIDE2', 'SHER-HAR-L', 'SHER-KIR-L', 'SOUTHSIDE', 'SQ.HILL1', 'SQ.HILL2', 'STRIPDIST', 'TAME-BEA-L', 'TAYLOR-L', 'TECHNOLOGY', 'UPTOWN1', 'UPTOWN2', 'W CIRC DR', 'WALT-WAR-L', 'WEST END']


We need some normalizations in order to join them!

#### 1.2. Fix the one-to-many problem (parking has subzones, payments have one zone name)

Some **payment zone names** are single areas, but the parking dataset splits them into multiple zone_ids.  
To make the join unambiguous (and avoid duplicating payment rows), we create a coarser key zone_group:

- UPTOWN1, UPTOWN2 → zone_group = "UPTOWN"
- SHADYSIDE1, SHADYSIDE2 → zone_group = "SHADYSIDE"
- SQ.HILL1, SQ.HILL2 → zone_group = "SQUIRREL HILL"
- HILL-DIST, HILL-DIST-2 → zone_group = "HILL DISTRICT"

For all other zones: zone_group = zone_id (no merge).

After this step, one payment zone corresponds to one parking group, so joining does not produce duplicated payment rows.

In [8]:
# Parking side:
# add zone_group to every parking zone_id
parking_zones_g = parking_zones.with_columns(
    pl.when(pl.col("zone_id").is_in(["UPTOWN1", "UPTOWN2"])).then(pl.lit("UPTOWN"))
    .when(pl.col("zone_id").is_in(["SHADYSIDE1", "SHADYSIDE2"])).then(pl.lit("SHADYSIDE"))
    .when(pl.col("zone_id").is_in(["SQ.HILL1", "SQ.HILL2"])).then(pl.lit("SQUIRREL HILL"))
    .when(pl.col("zone_id").is_in(["HILL-DIST", "HILL-DIST-2"])).then(pl.lit("HILL DISTRICT"))
    .otherwise(pl.col("zone_id"))
    .alias("zone_group")
)

# aggregate meters/locations counts at zone_group level ---
counts_group = (
    parking_zones_g
    .group_by("zone_group")
    .agg([
        pl.col("n_meters").sum().alias("n_meters"),
        pl.col("n_locations").sum().alias("n_locations"),
    ])
)

# aggregate to zone_group
locs_group = (
    parking_zones_g
    .select(["zone_group", "location_list"])
    .with_columns(
        # split "A, B, C" -> ["A"," B"," C"]
        pl.col("location_list")
          .str.split(",")
          .alias("location_item")
    )
    # take a list column and turn each element of the list into its own row
    .explode("location_item")
    # dedupe within each merged group
    .unique(subset=["zone_group", "location_item"])
    # aggregate back: list of unique locations + a readable string
    .group_by("zone_group")
    .agg([
        pl.col("location_item").sort().alias("location_unique_list"),
    ])
    .with_columns(
        pl.col("location_unique_list").list.join(", ").alias("location_unique_str")
    )
)

pz = (
    counts_group
    .join(locs_group.select(["zone_group", "location_unique_str"]), on="zone_group", how="left")
    .sort("zone_group")
)

# IMPORTANT we use this to map back
zone_group_to_zone_id = parking_zones_g.select(["zone_group","zone_id"]).unique()

pz.head()

zone_group,n_meters,n_locations,location_unique_str
str,i64,i64,str
"""18-CARSO-L""",2,1,"""18TH & CARSON LOT"""
"""18-SIDNE-L""",2,1,"""18TH & SIDNEY LOT"""
"""19-CARSO-L""",1,1,"""19TH & CARSON LOT"""
"""20-SIDNE-L""",2,1,"""20TH & SIDNEY LOT"""
"""42-BUTLE-L""",1,1,"""42ND & BUTLER LOT"""


In [9]:
# Payment transactions side:
def norm_text(expr):
    return (
        expr.cast(pl.Utf8)
        .str.to_uppercase()
        .str.replace_all(r"[().,/&-]", " ")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
        .str.replace_all("DISCTRICT", "DISTRICT")
    )

pt = pt.with_columns(
    norm_text(pl.col("zone_name")).alias("zone_name_norm")
)

# map the aggregated payment names into the same zone_group labels
pt = pt.with_columns(
    pl.when(pl.col("zone_name_norm") == "UPTOWN").then(pl.lit("UPTOWN"))
    .when(pl.col("zone_name_norm") == "SHADYSIDE").then(pl.lit("SHADYSIDE"))
    .when(pl.col("zone_name_norm") == "SQUIRREL HILL").then(pl.lit("SQUIRREL HILL"))
    .when(pl.col("zone_name_norm") == "HILL DISTRICT").then(pl.lit("HILL DISTRICT"))
    # otherwise keep the normalized name for now (we'll map remaining ones next step)
    .otherwise(pl.col("zone_name_norm"))
    .alias("zone_group")
)


pt.head()

zone,start,end,utc_start,transaction_type,amount,n_transaction,zone_code,zone_name,zone_name_norm,zone_group
str,str,str,str,str,f64,i64,i64,str,str,str
"""421 - NorthSide""","""2018-01-01T00:20:00""","""2018-01-01T00:30:00""","""2018-01-01T05:20:00""","""mobile""",4.0,1,421,"""NorthSide""","""NORTHSIDE""","""NORTHSIDE"""
"""403 - Uptown""","""2018-01-01T01:10:00""","""2018-01-01T01:20:00""","""2018-01-01T06:10:00""","""mobile""",3.0,1,403,"""Uptown""","""UPTOWN""","""UPTOWN"""
"""412 - East Liberty""","""2018-01-01T01:10:00""","""2018-01-01T01:20:00""","""2018-01-01T06:10:00""","""mobile""",3.0,1,412,"""East Liberty""","""EAST LIBERTY""","""EAST LIBERTY"""
"""421 - NorthSide""","""2018-01-01T01:20:00""","""2018-01-01T01:30:00""","""2018-01-01T06:20:00""","""mobile""",4.0,1,421,"""NorthSide""","""NORTHSIDE""","""NORTHSIDE"""
"""402 - Downtown 2""","""2018-01-01T06:00:00""","""2018-01-01T06:10:00""","""2018-01-01T11:00:00""","""mobile""",16.25,1,402,"""Downtown 2""","""DOWNTOWN 2""","""DOWNTOWN 2"""


In [10]:
# try first join
join1 = pt.join(pz,on="zone_group",how="inner")
join1_unique_zones = join1.select("zone_group").unique()
print(f"Number of joined zone group: {join1_unique_zones.height}")
print(join1_unique_zones.sort("zone_group").to_series().to_list())
pt_leftover1 = set(pt.select("zone_group").unique().to_series().to_list())\
                - set(join1_unique_zones.to_series().to_list())
print(f"\nNumber of zone group leftover in pt: {len(pt_leftover1)}")
print(sorted(pt_leftover1))
pz_leftover1 = set(pz.select("zone_group").unique().to_series().to_list())\
                - set(join1_unique_zones.to_series().to_list())
print(f"\nNumber of zone group leftover in pz: {len(pz_leftover1)}")
print(sorted(pz_leftover1))

Number of joined zone group: 12
['ALLENTOWN', 'BEECHVIEW', 'BROOKLINE', 'CARRICK', 'HILL DISTRICT', 'KNOXVILLE', 'NORTHSHORE', 'NORTHSIDE', 'SHADYSIDE', 'SQUIRREL HILL', 'UPTOWN', 'WEST END']

Number of zone group leftover in pt: 48
['18TH CARSON LOT', '18TH SIDNEY LOT', '19TH CARSON LOT', '20TH SIDNEY LOT', '42ND BUTLER LOT', '52ND BUTLER LOT', 'ANSLEY BEATTY LOT', 'ASTEROID WARRINGTON LOT', 'BAKERY SQ', 'BEACON BARTLETT LOT', 'BEECHVIEW LOT', 'BLOOMFIELD ON STREET', 'BROOKLINE LOT', 'BROWNSVILLE SANDKEY LOT', 'CENTRE CRAIG', 'DOUGLAS PHILLIPS LOT', 'DOWNTOWN 1', 'DOWNTOWN 2', 'EAST CARSON LOT', 'EAST LIBERTY', 'EAST OHIO STREET LOT', 'EVA BEATTY LOT', 'FORBES MURRAY LOT', 'FORBES SHADY LOT', 'FRIENDSHIP CEDARVILLE LOT', 'HARVARD BEATTY LOT', 'HOMEWOOD ZENITH LOT', 'IVY BELLEFONTE LOT', 'JCC FORBES LOT', 'LAWRENCEVILLE', 'MAIN ALEXANDER LOT', 'MELLON PARK', 'MT WASHINGTON', 'OAKLAND 1', 'OAKLAND 2', 'OAKLAND 3', 'OAKLAND 4', 'OBERSERVATORY HILL LOT', 'PENN CIRCLE NW LOT', 'SHERIDAN HA

#### 1.3. Normalize zone names so payment zones can join to parking zones

After creating zone_group, many zones still do not match because the two datasets use different naming styles:
  - Payment zones use human-readable names (e.g., "18th & Carson Lot", "Mt. Washington", "Technology Drive").
  - Parking zones use compact IDs/codes (e.g., 18-CARSO-L, MT.WASH, TECHNOLOGY).

**Fix:** Create a shared normalized name key zone_group_norm on both datasets.
- Payment side: normalize the extracted zone_group text (uppercase, fix typos, convert ordinals like 18TH→18, standardize punctuation).
- Parking side: map compact zone_group codes to canonical names (via a lookup table), then apply the same normalization.
- Join using zone_group_norm.

Example:
- Payment "18th & Carson Lot" → zone_group_norm = "18 CARSON LOT"
- Parking 18-CARSO-L → canonical "18 CARSON LOT" → zone_group_norm = "18 CARSON LOT"
- Join on zone_group_norm

In [11]:
def norm_zone(expr):
    return (
        expr.cast(pl.Utf8)
        .str.to_uppercase()
        # ordinals: 18TH -> 18, 42ND -> 42, etc.
        .str.replace_all(r"\b(\d+)(ST|ND|RD|TH)\b", r"$1")
        # unify separators
        .str.replace_all(r"[&/]", " ")
        .str.replace_all(r"[().,/]", " ")
        # normalize LOT wording (keep LOT because parking codes represent lots)
        .str.replace_all(r"\bON\s+STREET\b", "")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
        .str.replace_all("DISCTRICT", "DISTRICT")
    )

# Payment side: create zone_group_norm from pt.zone_group
pt_normed = pt.with_columns(
    norm_zone(pl.col("zone_group")).alias("zone_group_norm")
)

pt_normed.select(["zone_group","zone_group_norm"]).unique().sort("zone_group_norm").head()

zone_group,zone_group_norm
str,str
"""18TH CARSON LOT""","""18 CARSON LOT"""
"""18TH SIDNEY LOT""","""18 SIDNEY LOT"""
"""19TH CARSON LOT""","""19 CARSON LOT"""
"""20TH SIDNEY LOT""","""20 SIDNEY LOT"""
"""42ND BUTLER LOT""","""42 BUTLER LOT"""


In [12]:
# Parking side: build a mapping table from compact codes -> names
"""
    pz.zone_group contains zone_group like:
        "18-CARSO-L", "MT.WASH", "TECHNOLOGY", ...
    These will NOT match payment names by simple normalization.
    So we create zone_group_map: a lookup that translates each code into
    a "zone_group_name", and then normalize it too.
"""
zone_group_map = pl.DataFrame({
    "zone_group": [
        # lots / streets (code -> name)
        "18-CARSO-L","18-SIDNE-L","19-CARSO-L","20-SIDNE-L","42-BUTLE-L","5224BUTL-L",
        "ANSL-BEA-L","ASTE-WAR-L","BAKERY-SQ", "BEAC-BAR-L","BEECHVIE-L","BROW-SAN-L","BROOKLIN-L",
        "DOUG-PHI-L","EASTCARS-L","EASTOHIO-L","EVA-BEAT-L","FORB-MUR-L","FORB-SHA-L",
        "FRIE-CED-L","HOME-ZEN-L","IVY-BELL-L","JCC-L","LAWRENCEV","MAIN-ALE-L",
        "MELONPARK","MT.WASH","OBSERHIL-L","PENNC.NW-L","SHER-HAR-L","SHER-KIR-L",
        "TAME-BEA-L","TAYLOR-L","TECHNOLOGY","STRIPDIST","SOUTHSIDE","WALT-WAR-L",
        # already close but formatting differs
        "DOWNTOWN1","DOWNTOWN2","OAKLAND1","OAKLAND2","OAKLAND3","OAKLAND4",
        "EASTLIB",
    ],
    "zone_group_name": [
        "18 CARSON LOT","18 SIDNEY LOT","19 CARSON LOT","20 SIDNEY LOT","42 BUTLER LOT","52 BUTLER LOT",
        "ANSLEY BEATTY LOT","ASTEROID WARRINGTON LOT","BAKERY SQ", "BEACON BARTLETT LOT","BEECHVIEW LOT","BROWNSVILLE SANDKEY LOT","BROOKLINE LOT",
        "DOUGLAS PHILLIPS LOT","EAST CARSON LOT","EAST OHIO STREET LOT","EVA BEATTY LOT","FORBES MURRAY LOT","FORBES SHADY LOT",
        "FRIENDSHIP CEDARVILLE LOT","HOMEWOOD ZENITH LOT","IVY BELLEFONTE LOT","JCC FORBES LOT","LAWRENCEVILLE","MAIN ALEXANDER LOT",
        "MELLON PARK","MT WASHINGTON","OBERSERVATORY HILL LOT","PENN CIRCLE NW LOT","SHERIDAN HARVARD LOT","SHERIDAN KIRKWOOD LOT",
        "TAMELLO BEATTY LOT","TAYLOR STREET LOT","TECHNOLOGY DRIVE","STRIP DISTRICT","SS SSW","WALTER WARRINGTON LOT",
        
        "DOWNTOWN 1","DOWNTOWN 2","OAKLAND 1","OAKLAND 2","OAKLAND 3","OAKLAND 4",
        "EAST LIBERTY",
    ]
}).with_columns(
    # normalize exactly the same way as payments
    norm_zone(pl.col("zone_group_name")).alias("zone_group_norm")
).select(["zone_group","zone_group_norm"])


# For parking groups listed in zone_group_map, use the mapped normalized name.
# For any other group NOT in the mapping, fall back to normalizing the code itself.
# (This fallback helps for zones that already look like names: ALLENTOWN, CARRICK, etc.)
pz = pz.with_columns(
    norm_zone(pl.col("zone_group")).alias("zone_group_norm_fallback")
)
pz_normed = (
    pz.join(zone_group_map, on="zone_group", how="left")
      .with_columns(
          pl.coalesce([pl.col("zone_group_norm"), pl.col("zone_group_norm_fallback")]).alias("zone_group_norm")
      )
      .drop("zone_group_norm_fallback")
)

pz_normed.select(["zone_group","zone_group_norm"]).unique().sort("zone_group_norm").head()

zone_group,zone_group_norm
str,str
"""18-CARSO-L""","""18 CARSON LOT"""
"""18-SIDNE-L""","""18 SIDNEY LOT"""
"""19-CARSO-L""","""19 CARSON LOT"""
"""20-SIDNE-L""","""20 SIDNEY LOT"""
"""42-BUTLE-L""","""42 BUTLER LOT"""


In [13]:
join2 = pt_normed.join(pz_normed, on="zone_group_norm", how="inner")

join2_unique = join2.select("zone_group_norm").unique()
print("Nomber of joined zone_group_norm:", join2_unique.height)

# leftovers check
pt_left2 = set(pt_normed.select("zone_group_norm").unique().to_series().to_list()) - set(join2_unique.to_series().to_list())
pz_left2 = set(pz_normed.select("zone_group_norm").unique().to_series().to_list()) - set(join2_unique.to_series().to_list())

print("\nLeftover in pt:", len(pt_left2), sorted(pt_left2))
print("Leftover in pz:", len(pz_left2), sorted(pz_left2))

Nomber of joined zone_group_norm: 57

Leftover in pt: 3 ['CENTRE CRAIG', 'HARVARD BEATTY LOT', 'SHILOH STREET LOT']
Leftover in pz: 1 ['W CIRC DR']


Great!

For the remaining 3 payment leftovers likely require looking into location_unique_str because the zone_id list doesn’t have obvious counterparts.

But for the time constraint let's stop here.

### 2. Payments Transaction & Space Counts

Let's try to join space_counts and payment transactions

In [14]:
print(f"Number of unique zones in space_counts: {space_counts.select("zone").unique().height}")
print(space_counts.select("zone").unique().to_series().to_list())

Number of unique zones in space_counts: 67
['345 - 20th & Sidney Lot', '351 - Brownsville & Sandkey Lot', '343 - 19th & Carson Lot', '415 - SS & SSW', '416 - Carrick', '307 - Eva Beatty Lot', 'SHADYSIDE2', '302 - Sheridan Kirkwood Lot', 'SQ.HILL2', '423 - West End', '407 - Oakland 1', '414 - Mellon Park', '304 - Tamello Beatty Lot', '323 - Douglas Phillips Lot', '325 - JCC/Forbes Lot', '337 - 52nd & Butler Lot', '355 - Asteroid Warrington Lot', '403 - Uptown', '334 - Taylor Street Lot', '342 - East Carson Lot', '401 - Downtown 1', '420 - Mt. Washington', '321 - Beacon Bartlett Lot', '341 - 18th & Sidney Lot', '338 - 42nd & Butler Lot', 'W CIRC DR', '324 - Forbes Murray Lot', '419 - Brookline', '418 - Beechview', '424 - Technology Drive', '369 - Main/Alexander Lot', '409 - Oakland 3', '311 - Ansley Beatty Lot', '405 - Lawrenceville', '363 - Beechview Lot', '331 - Homewood Zenith Lot', '361 - Brookline Lot', 'UPTOWN1', '335 - Friendship Cedarville Lot', '314 - Penn Circle NW Lot', '328 -

In [15]:
# try joining directly
join3 = pt_normed.join(space_counts, on="zone",how="inner")
join3_unique = join3.select("zone").unique()
print(f"Number of joined zone: {join3_unique.height}")

# leftovers check
pt_left3 = set(pt_normed.select("zone").unique().to_series().to_list()) - set(join3_unique.to_series().to_list())
sc_left3 = set(space_counts.select("zone").unique().to_series().to_list()) - set(join3_unique.to_series().to_list())
print("\nLeftover in pt:", len(pt_left3), sorted(pt_left3))
print("Leftover in sc:", len(sc_left3), sorted(sc_left3))

Number of joined zone: 57

Leftover in pt: 3 ['308 - Harvard Beatty Lot', '329 - Centre Craig', '427 - Knoxville']
Leftover in sc: 10 ['HILL-DIST-2', 'S. Craig', 'SHADYSIDE1', 'SHADYSIDE2', 'SQ.HILL1', 'SQ.HILL2', 'Southside Lots', 'UPTOWN1', 'UPTOWN2', 'W CIRC DR']


The result is pretty good we have 57 joined zone! (it could be improved by joining sc to parking_meters, or grouping subzones. Let's leave it for future works).

### 3. Saving final datasets

To avoid redoing string-cleaning and to make future joins consistent, save the following “final” tables and crosswalks:

**parking_zones_final (zone_group-level)**
- Includes: zone_group, zone_group_norm, n_meters, n_locations, location_unique_str.

**zone_group_norm_to_zone_id (crosswalk)**
- Maps merged zones back to original parking zone ids:
  zone_group_norm → zone_id
- Used to join zone-level results to other parking tables keyed by zone_id (rates/hours/meters).

**payment_transactions_final**
- Payment transactions with a clean join key:
  - keep original fields + add zone_group_norm
- Used to join payments to parking zones consistently.

**parking_spaces_final**
- The original space_counts table +.zone_group_norm.
- Used to join parking_spaces_final to payment transactions and parking zones using the same normalized zone key.

#### 3.1 Final Datasets Preparation

**parking_zones_final**

In [16]:
# prepare paking_zones_final
parking_zones_final = pz_normed.select(["zone_group_norm"\
                        ,"n_meters","n_locations","location_unique_str"])\
                        .rename({"location_unique_str":"location_list"})
parking_zones_final.head()

zone_group_norm,n_meters,n_locations,location_list
str,i64,i64,str
"""18 CARSON LOT""",2,1,"""18TH & CARSON LOT"""
"""18 SIDNEY LOT""",2,1,"""18TH & SIDNEY LOT"""
"""19 CARSON LOT""",1,1,"""19TH & CARSON LOT"""
"""20 SIDNEY LOT""",2,1,"""20TH & SIDNEY LOT"""
"""42 BUTLER LOT""",1,1,"""42ND & BUTLER LOT"""


**zone_group_to_zone_id**

In [17]:
# take group norm from pz_normed
print(pz_normed.columns)

# print("\nNumber of zone id in parking zones",parking_zones.select("zone_id").unique().height)
# zone_group_to_zone_id is saved before
zone_group_norm_to_zone_id = zone_group_to_zone_id.join(pz_normed, on="zone_group")\
                            .unique().select(["zone_id","zone_group_norm"])
zone_group_norm_to_zone_id.head()

['zone_group', 'n_meters', 'n_locations', 'location_unique_str', 'zone_group_norm']


zone_id,zone_group_norm
str,str
"""5224BUTL-L""","""52 BUTLER LOT"""
"""SQ.HILL2""","""SQUIRREL HILL"""
"""18-CARSO-L""","""18 CARSON LOT"""
"""CARRICK""","""CARRICK"""
"""TAYLOR-L""","""TAYLOR STREET LOT"""


**payment_transactions_final**

In [18]:
payment_transactions_final = pt_normed.select([
    'zone', 'start', 'end', 'utc_start', 
    'transaction_type', 'n_transaction', 'amount', 
    'zone_group_norm'
])
payment_transactions_final.head()

zone,start,end,utc_start,transaction_type,n_transaction,amount,zone_group_norm
str,str,str,str,str,i64,f64,str
"""421 - NorthSide""","""2018-01-01T00:20:00""","""2018-01-01T00:30:00""","""2018-01-01T05:20:00""","""mobile""",1,4.0,"""NORTHSIDE"""
"""403 - Uptown""","""2018-01-01T01:10:00""","""2018-01-01T01:20:00""","""2018-01-01T06:10:00""","""mobile""",1,3.0,"""UPTOWN"""
"""412 - East Liberty""","""2018-01-01T01:10:00""","""2018-01-01T01:20:00""","""2018-01-01T06:10:00""","""mobile""",1,3.0,"""EAST LIBERTY"""
"""421 - NorthSide""","""2018-01-01T01:20:00""","""2018-01-01T01:30:00""","""2018-01-01T06:20:00""","""mobile""",1,4.0,"""NORTHSIDE"""
"""402 - Downtown 2""","""2018-01-01T06:00:00""","""2018-01-01T06:10:00""","""2018-01-01T11:00:00""","""mobile""",1,16.25,"""DOWNTOWN 2"""


**zone_to_zone_group_norm**

In [19]:
# prepare zone_to_zone_group_norm
# get group norm from pt_normed
print(pt_normed.columns)
zone_to_zone_group_norm = pt_normed.select(["zone","zone_group_norm"]).unique()
zone_to_zone_group_norm.head()

['zone', 'start', 'end', 'utc_start', 'transaction_type', 'amount', 'n_transaction', 'zone_code', 'zone_name', 'zone_name_norm', 'zone_group', 'zone_group_norm']


zone,zone_group_norm
str,str
"""421 - NorthSide""","""NORTHSIDE"""
"""301 - Sheridan Harvard Lot""","""SHERIDAN HARVARD LOT"""
"""322 - Forbes Shady Lot""","""FORBES SHADY LOT"""
"""420 - Mt. Washington""","""MT WASHINGTON"""
"""403 - Uptown""","""UPTOWN"""


In [20]:
# parking_spaces_final = space_counts + zone_group_norm
parking_spaces_final = space_counts.select([
    "zone", "spaces", "type"
]).join(zone_to_zone_group_norm,on="zone")
parking_spaces_final.head()

zone,spaces,type,zone_group_norm
str,i64,str,str
"""301 - Sheridan Harvard Lot""",41,"""off-street""","""SHERIDAN HARVARD LOT"""
"""302 - Sheridan Kirkwood Lot""",114,"""off-street""","""SHERIDAN KIRKWOOD LOT"""
"""304 - Tamello Beatty Lot""",76,"""off-street""","""TAMELLO BEATTY LOT"""
"""307 - Eva Beatty Lot""",130,"""off-street""","""EVA BEATTY LOT"""
"""311 - Ansley Beatty Lot""",23,"""off-street""","""ANSLEY BEATTY LOT"""


#### 3.2 Final Check

In [21]:
# try joining parking_zones to space_counts
n_pz = parking_zones.height
n_sc = space_counts.height
pz_sc = parking_zones_final.join(parking_spaces_final, on="zone_group_norm")
n_pz_sc = pz_sc.height
print(f"Number of row in parking zones: {n_pz}")
print(f"Number of row in space counts: {n_sc}")
print(f"Number of row in parking zone join space counts: {n_pz_sc}")

Number of row in parking zones: 62
Number of row in space counts: 67
Number of row in parking zone join space counts: 56


In [22]:
# try joining all three datasets
n_pt = payment_transactions.height
pz_sc_pt = parking_zones_final.join(parking_spaces_final, on="zone_group_norm")\
        .join(payment_transactions_final, on="zone_group_norm")
n_pz_sc_pt = pz_sc_pt.height
print(f"Number of row in parking zones: {n_pz}")
print(f"Number of row in space counts: {n_sc}")
print(f"Number of row in payment transaction: {n_pt}")
print(f"Number of row in pz_sc_pt: {n_pz_sc_pt}")

Number of row in parking zones: 62
Number of row in space counts: 67
Number of row in payment transaction: 3383545
Number of row in pz_sc_pt: 3282460


Great! We successfully joined most of the rows

#### 3.3. Saving

In [23]:
# Save datasets
parking_zones_final.write_csv("../final_datasets/parking_zones.csv")
zone_group_norm_to_zone_id.write_csv("../final_datasets/zone_group_norm_to_zone_id.csv")
payment_transactions_final.write_csv("../final_datasets/payment_transactions.csv")
parking_spaces_final.write_csv("../final_datasets/parking_spaces_zone_group_norm.csv")

In [ ]:
# Save other parking datasets
parking_meters = pl.read_csv("cleaned_data/parking_meter_data/parking_meters.csv")
parking_rates = pl.read_csv("cleaned_data/parking_meter_data/parking_rates.csv")

parking_meters.write_csv("../final_datasets/parking_meters.csv")
parking_rates.write_csv("../final_datasets/parking_rates.csv")